# Discussion 11/5/2026

In this discussion we compare the optimization methods from lecture: SGD, gradient descent with momentum, Adam, Newton's method, and BFGS.

You can use the Shared Computing Cluster (SCC) or Google Colab to run this notebook.

The general instructions for running on the SCC are available under General Resources on [Piazza](https://piazza.com/bu/fall2026/ds722/resources).

## Problem: Optimization

We want to find the minimum of the function

$$
f(x,y) = -20 e^{-0.2 \sqrt{0.5 (x^{2} + y^{2})}}
- e^{0.5 \left(\cos(2 \pi x) + \cos(2 \pi y)\right)}
+ e + 20.
$$

This is called the Ackley function. Its global minimum is at $(0, 0)$, where $f(0, 0) = 0$.

Run the next two cells. The first one plots the function. The second one defines the functions you will use. You do not need to change them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

def f(z):
    x, y = z[0], z[1]
    return (-20 * torch.exp(-0.2 * torch.sqrt(0.5 * (x**2 + y**2)))
            - torch.exp(0.5 * (torch.cos(2 * np.pi * x) + torch.cos(2 * np.pi * y))) + np.e + 20)

# Evaluate f on a grid for plotting
xy = np.linspace(-2, 2, 300)
X, Y = np.meshgrid(xy, xy)
Z = f(torch.tensor(np.array([X, Y]))).numpy()

fig = plt.figure(figsize=(14, 6))

ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(X, Y, Z, cmap='viridis')
ax1.set_xlabel('$x$')
ax1.set_ylabel('$y$')
ax1.set_title('3-D plot of the Ackley function')

ax2 = fig.add_subplot(1, 2, 2)
contour = ax2.contourf(X, Y, Z, levels=50, cmap='viridis')
fig.colorbar(contour, ax=ax2)
ax2.set_xlabel('$x$')
ax2.set_ylabel('$y$')
ax2.set_title('Contour plot of the Ackley function')
plt.show()

In [ ]:
def run_optimizer(optimizer, x0, steps=1000, **settings):
    # Runs a PyTorch optimizer (e.g., torch.optim.SGD) starting from the point x0
    x = torch.nn.Parameter(torch.tensor(x0, dtype=torch.float64))
    if optimizer is torch.optim.LBFGS:
        settings['max_iter'] = 1  # one BFGS step per loop iteration
    opt = optimizer([x], **settings)

    def closure():
        opt.zero_grad()
        loss = f(x)
        loss.backward()
        return loss

    path = [x.detach().numpy().copy()]
    for k in range(steps):
        opt.step(closure)
        path.append(x.detach().numpy().copy())
        if np.linalg.norm(path[-1]) < 1e-5:  # stop when we are within 1e-5 of (0, 0)
            break

    print(f"Steps: {len(path) - 1}, final point: {path[-1].round(5)}, distance from (0, 0): {np.linalg.norm(path[-1]):.1e}")
    return np.array(path)

def newton_method(x0, steps=10, lr=1.0):
    # Runs Newton's method starting from the point x0 with step size lr
    x = torch.tensor(x0, dtype=torch.float64)
    path = [x.numpy().copy()]
    for k in range(steps):
        z = x.clone().requires_grad_(True)
        gradient = torch.autograd.grad(f(z), z)[0]
        hessian = torch.autograd.functional.hessian(f, x)
        x = x - lr*torch.linalg.solve(hessian, gradient)
        path.append(x.numpy().copy())

    print(f"Steps: {steps}, final point: {path[-1].round(5)}, distance from (0, 0): {np.linalg.norm(path[-1]):.1e}")
    return np.array(path)

def plot_paths(paths):
    # paths is a dictionary {name: path}
    plt.figure(figsize=(8, 6))
    plt.contourf(X, Y, Z, levels=50, cmap='viridis')
    plt.colorbar()
    for name, path in paths.items():
        line, = plt.plot(path[:, 0], path[:, 1], '.-', label=name)
        plt.scatter(path[-1, 0], path[-1, 1], color=line.get_color(), edgecolors='black', s=150, zorder=3)  # end point
    plt.scatter(0, 0, c='white', edgecolors='black', s=100, label='Global minimum')
    plt.xlabel('$x$')
    plt.ylabel('$y$')
    plt.legend()
    plt.show()

**How to use these functions**

`run_optimizer(optimizer, x0, steps=1000, **settings)` runs one optimization method and returns its path (the list of points it visited).

- `optimizer`: the PyTorch method to use, e.g., `torch.optim.SGD`, `torch.optim.Adam`, or `torch.optim.LBFGS`.
- `x0`: the starting point, e.g., `[0.6, 0.4]`.
- `steps`: the maximum number of steps (default 1000). The method stops early if it gets within $10^{-5}$ of $(0, 0)$.
- `**settings`: any settings for the method, e.g., the learning rate `lr=0.01` or `momentum=0.9`.

It prints the number of steps taken, the final point, and the distance from the final point to $(0,0)$.

`newton_method(x0, steps=10, lr=1.0)` runs Newton's method and returns its path.

- `x0`: the starting point, e.g., `[0.6, 0.4]`.
- `steps`: the number of steps (default 10).
- `lr`: the learning rate, which scales each Newton step (default 1.0, the standard Newton step), e.g., `lr=0.5`.

It prints the same things as `run_optimizer`.

`plot_paths(paths)` draws paths on the contour plot. The input `paths` is a dictionary where each key is a name for the legend and each value is a path returned by `run_optimizer` or `newton_method`, e.g., `{'SGD': path_sgd, 'Adam': path_adam}`. The large dot on each path shows where the method ended.

### Step 1

Look at the contour plot. Does $f$ have one minimum or many? Why could this be a problem for gradient descent?

*Your answer here.*

### Step 2

Starting from the point $(0.6, 0.4)$, run SGD, gradient descent with momentum, and Adam. Here is how to run SGD:

```python
path_sgd = run_optimizer(torch.optim.SGD, [0.6, 0.4], lr=0.01)
```

In lecture, SGD estimates the gradient using a random batch of data. Our function has no data, so here SGD is the same as gradient descent.

For momentum use `torch.optim.SGD` with the extra setting `momentum=0.9`. For Adam use `torch.optim.Adam` with `lr=0.01`. Then plot the three paths with

```python
plot_paths({'SGD': path_sgd, 'Momentum': path_momentum, 'Adam': path_adam})
```

Which method gets closest to $(0, 0)$?

Now try Adam again with `lr=0.002`. Does it reach $(0, 0)$ now?

In [ ]:
# TODO

*Your answer here.*

### Step 3

Now try the second order methods from $(0.6, 0.4)$:

```python
path_newton = newton_method([0.6, 0.4])
path_bfgs = run_optimizer(torch.optim.LBFGS, [0.6, 0.4], lr=1, line_search_fn='strong_wolfe')
```

(PyTorch's `LBFGS` is a version of BFGS that uses less memory.) Plot both paths.

Did Newton's method find the minimum? Look at where it ended up on the contour plot. Is that point a minimum? Did BFGS find the minimum? If BFGS does not reach $(0, 0)$ on your computer, try a smaller learning rate, like $0.5$, $0.1$, or $0.01$.

In [ ]:
# TODO

*Your answer here.*